In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [5]:
import pandas as pd

df = pd.read_csv("/home/aleksandr/EV_prediction_service/tmp/train(2).csv")

In [8]:
df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [11]:
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 50)

data = df.copy()
data.columns = data.columns.str.strip()
data = data.replace(r"^\s*$", np.nan, regex=True)

TARGET = "Will_Buy_EV"
ID = "id"

assert TARGET in data.columns, f"Нет колонки {TARGET}"

numeric_candidates = [
    "Age", "Annual_Income_USD", "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
    "Environmental_Concern_Level", "Range_Anxiety_Level",
]

# Эти признаки считаем категориями даже при числовой кодировке.
categorical_candidates = [
    "Gender", "City_Type", "Current_Car_Type",
    "Home_Charging_Possible", "Subsidy_Available",
]

# Числовые строки преобразуем, только если все непустые значения распознаны.
for col in numeric_candidates:
    if col not in data.columns:
        continue
    converted = pd.to_numeric(data[col], errors="coerce")
    invalid = data[col].notna() & converted.isna()
    if not invalid.any():
        data[col] = converted
    else:
        print(f"{col}: оставлена исходная колонка; "
              f"нечисловые значения: {data.loc[invalid, col].unique()[:5]}")

feature_cols = [c for c in data.columns if c not in [ID, TARGET]]
num_cols = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(data[c])
    and not pd.api.types.is_bool_dtype(data[c])
    and c not in categorical_candidates
]
cat_cols = [c for c in feature_cols if c not in num_cols]

# Бесконечности учитываем отдельно, затем заменяем на NaN в рабочей копии.
inf_counts = data[num_cols].isin([np.inf, -np.inf]).sum()
if inf_counts.any():
    print("Бесконечности, заменённые на NaN:")
    display(inf_counts[inf_counts > 0])
    data[num_cols] = data[num_cols].replace([np.inf, -np.inf], np.nan)

# Небольшие дискретные шкалы дополнительно посмотрим как категории.
discrete_cols = [c for c in num_cols if 1 <= data[c].nunique() <= 10]
cat_view_cols = cat_cols + discrete_cols

def plot_grid(columns, draw, ncols=3, height=3.7):
    if not columns:
        print("Нет подходящих признаков.")
        return

    ncols = min(ncols, len(columns))
    nrows = int(np.ceil(len(columns) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(5.5 * ncols, height * nrows), squeeze=False
    )
    for ax, col in zip(axes.flat, columns):
        draw(col, ax)
        ax.set_title(col)
    for ax in axes.flat[len(columns):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

print("Числовые:", num_cols)
print("Категориальные:", cat_cols)

Range_Anxiety_Level: оставлена исходная колонка; нечисловые значения: <StringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str
Числовые: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Категориальные: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [12]:
print(f"Строк: {len(data):,}; колонок: {data.shape[1]}")
display(data.head())

quality = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing_n": data.isna().sum(),
    "missing_%": data.isna().mean().mul(100).round(2),
    "unique_n": data.nunique(dropna=True),
}).sort_values("missing_%", ascending=False)

display(quality)

print("Полные дубликаты строк:", data.duplicated().sum())
print(
    "Повторяющиеся строки без id:",
    data.drop(columns=[ID], errors="ignore").duplicated().sum()
)
# Совпадение признаков у разных людей само по себе не означает ошибку.

if ID in data.columns:
    print("Повторные непустые id:", data[ID].dropna().duplicated().sum())

print("Константы и полностью пустые колонки:",
      quality.index[quality["unique_n"] <= 1].tolist())

missing = quality.loc[quality["missing_n"] > 0, "missing_%"]
if len(missing):
    missing.sort_values().plot.barh(figsize=(9, max(3, len(missing) * 0.4)))
    plt.xlabel("Пропуски, %")
    plt.title("Пропуски по колонкам")
    plt.tight_layout()
    plt.show()
else:
    print("Пропусков нет.")

if num_cols:
    display(
        data[num_cols]
        .describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])
        .T.round(2)
    )

if cat_cols:
    display(data[cat_cols].astype("string").describe().T)

Строк: 668,665; колонок: 15


,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


,dtype,missing_n,missing_%,unique_n
id,int64,0,0.0,668665
Age,int64,0,0.0,45
Annual_Income_USD,float64,0,0.0,13214
Daily_Commute_km,float64,0,0.0,805
Number_of_Cars_Owned,int64,0,0.0,4
Charging_Stations_Near_Home,int64,0,0.0,15
Charging_Stations_Near_Work,int64,0,0.0,20
Environmental_Concern_Level,float64,0,0.0,5
Gender,str,0,0.0,3
City_Type,str,0,0.0,3


Полные дубликаты строк: 0
Повторяющиеся строки без id: 0
Повторные непустые id: 0
Константы и полностью пустые колонки: []
Пропусков нет.


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Age,668665.0,47.04,12.88,25.0,25.0,27.0,36.0,47.0,58.0,67.0,69.0,69.0
Annual_Income_USD,668665.0,84769.27,28648.03,30000.0,30000.0,30000.0,67376.0,84880.0,102753.0,132902.0,151159.0,188549.0
Daily_Commute_km,668665.0,32.16,18.73,5.0,5.0,5.0,17.2,33.6,47.4,59.8,68.5,98.7
Number_of_Cars_Owned,668665.0,1.71,0.73,1.0,1.0,1.0,1.0,2.0,2.0,3.0,4.0,4.0
Charging_Stations_Near_Home,668665.0,4.96,3.93,0.0,0.0,0.0,2.0,4.0,7.0,13.0,14.0,14.0
Charging_Stations_Near_Work,668665.0,7.18,5.19,0.0,0.0,1.0,3.0,6.0,10.0,18.0,19.0,19.0
Environmental_Concern_Level,668665.0,2.94,1.43,1.0,1.0,1.0,2.0,3.0,4.0,5.0,5.0,5.0


,count,unique,top,freq
Gender,668665,3,Male,367954
City_Type,668665,3,Urban,289305
Current_Car_Type,668665,4,Sedan,303459
Home_Charging_Possible,668665,2,Yes,462677
Subsidy_Available,668665,2,Yes,419909
Range_Anxiety_Level,668665,3,Low,603972


In [13]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

TARGET = "Will_Buy_EV"
SEED = 42

normalized = (
    data[TARGET].astype("string")
    .str.strip().str.lower()
    .replace("", pd.NA)
)
encoded = normalized.map({
    "yes": 1, "no": 0,
    "1": 1, "0": 0,
    "1.0": 1, "0.0": 0,
})

unknown = normalized.notna() & encoded.isna()
if unknown.any():
    raise ValueError(
        f"Неизвестные значения target: {normalized[unknown].unique().tolist()}"
    )

data[TARGET] = encoded.astype("Int64")
print("Строк без target, исключённых из обучения:", data[TARGET].isna().sum())

model_data = data.loc[data[TARGET].notna()].copy()
X = model_data.drop(columns=[TARGET, "id"], errors="ignore")
y = model_data[TARGET].astype(int)

if y.nunique() != 2:
    raise ValueError("Для бинарной классификации нужны оба класса: 0 и 1.")

known_categories = [
    "Gender", "City_Type", "Current_Car_Type",
    "Home_Charging_Possible", "Subsidy_Available",
]
cat_features = [
    c for c in X.columns
    if c in known_categories
    or not pd.api.types.is_numeric_dtype(X[c])
    or pd.api.types.is_bool_dtype(X[c])
]

for col in cat_features:
    X[col] = X[col].astype("string").fillna("__MISSING__").astype(str)

for col in X.columns.difference(cat_features):
    X[col] = X[col].astype(float).replace([np.inf, -np.inf], np.nan)

print("Категориальные признаки:", cat_features)
display(y.value_counts().sort_index().rename("count").to_frame())

Строк без target, исключённых из обучения: 0
Категориальные признаки: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


,count
Will_Buy_EV,
0,551886
1,116779


In [14]:

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid,
    test_size=0.25,
    stratify=y_train_valid,
    random_state=SEED,
)

if any(part.nunique() != 2 for part in [y_train, y_valid, y_test]):
    raise ValueError(
        "Недостаточно примеров для присутствия обоих классов во всех выборках."
    )

display(pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "Yes_share": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"]).round(3))

,rows,Yes_share
train,401199,0.175
validation,133733,0.175
test,133733,0.175


In [15]:
model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    allow_writing_files=False,
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100,
    use_best_model=True,
    verbose=100,
)

print("Деревьев в сохранённой модели:", model.tree_count_)
print("Лучший validation ROC-AUC:",
      round(model.get_best_score()["validation"]["AUC"], 4))

0:	test: 0.9305526	best: 0.9305526 (0)	total: 180ms	remaining: 4m 29s
100:	test: 0.9394477	best: 0.9394477 (100)	total: 8.85s	remaining: 2m 2s
200:	test: 0.9399834	best: 0.9399834 (200)	total: 17.6s	remaining: 1m 53s
300:	test: 0.9403934	best: 0.9403934 (300)	total: 26.6s	remaining: 1m 45s
400:	test: 0.9407443	best: 0.9407443 (400)	total: 35.7s	remaining: 1m 37s
500:	test: 0.9409569	best: 0.9409569 (500)	total: 44.9s	remaining: 1m 29s
600:	test: 0.9410700	best: 0.9410700 (600)	total: 54.3s	remaining: 1m 21s
700:	test: 0.9411573	best: 0.9411587 (696)	total: 1m 3s	remaining: 1m 12s
800:	test: 0.9412492	best: 0.9412492 (800)	total: 1m 12s	remaining: 1m 3s
900:	test: 0.9412828	best: 0.9412840 (894)	total: 1m 22s	remaining: 54.6s
1000:	test: 0.9413099	best: 0.9413099 (1000)	total: 1m 31s	remaining: 45.7s
1100:	test: 0.9413366	best: 0.9413374 (1094)	total: 1m 40s	remaining: 36.6s
1200:	test: 0.9413520	best: 0.9413535 (1193)	total: 1m 50s	remaining: 27.4s
1300:	test: 0.9413651	best: 0.9413651

In [16]:
bundle = {
    "pipeline": model,
    "metadata": {
        "model_version": "1.0",
        "features": X_train.columns.tolist(),
        "threshold": 0.5
    }
}

In [19]:
import joblib

joblib.dump(bundle, "/home/aleksandr/EV_prediction_service/artifacts/model.joblib")

loaded_model = joblib.load("/home/aleksandr/EV_prediction_service/artifacts/model.joblib")